# DNN price forecasting on Colab

Runs the epftoolbox DNN (Lago, Marcjasz, De Schutter & Weron 2021) on a
bidding zone, using Colab's GPU.

**Set the runtime first:** Runtime → Change runtime type → GPU.

### Read this before you rely on the GPU

This workload is not compute-bound. Measured on one real recalibration:
218 epochs at 0.36 s each, of which **60% is the per-epoch validation
pass**, on a 224→216→24 network with roughly 1,100 training samples. The
matrix multiplies are trivial; the cost is per-call overhead in the custom
training loop, which a faster device does not remove. Expect a useful
speedup, not a tenfold one, and measure it below rather than assuming it.

## 1. Setup

The runtime is wiped on disconnect, so everything worth keeping goes to Drive.

In [ ]:
!git clone --branch claude/entsoe-api-test-script-1cl1kd https://github.com/Freddy5445/EPF_Masters.git
%cd EPF_Masters

In [ ]:
import colab_setup

# Installs what Colab lacks, installs the vendored epftoolbox from the local
# tree, mounts Drive, and reports what TensorFlow can actually use.
has_gpu = colab_setup.bootstrap()

## 2. Point at your data

`datasets/*.csv` is gitignored, so the clone has no data in it. Put the
zone CSV on Drive and point `DATASETS` at it.

Produce that CSV locally with:

```
python run_lear_from_clean.py --zone DK1 --exog load-windsolar --csv-only
```

and copy `datasets/DK1_clean_load-windsolar.csv` to Drive. The DNN reads the
same epftoolbox-layout CSV the LEAR runs use, so both models see identical
inputs — which is what makes their scores comparable.

In [ ]:
import os

DRIVE = '/content/drive/MyDrive/EPF_Masters'
DATASETS = os.path.join(DRIVE, 'datasets')
OUT      = os.path.join(DRIVE, 'experiments')
DATASET  = 'DK1_clean_load-windsolar'

os.makedirs(DATASETS, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('datasets:', os.listdir(DATASETS) or '(empty - upload the CSV here)')

### Or upload the CSV straight from your machine

Run this cell only if the file is not already on Drive.

In [ ]:
from google.colab import files
import shutil

for name in files.upload():
    shutil.move(name, os.path.join(DATASETS, name))
    print('->', os.path.join(DATASETS, name))

## 3. Smoke test first

Five hyperparameter evaluations and three forecast days. This proves the
pipeline runs on this runtime. It says nothing about accuracy — five
evaluations is not a search.

In [ ]:
# Built as a list rather than a ! shell line, because the paths are
# Python variables and a shell cell cannot see them.
import subprocess, sys

cmd = [sys.executable, 'run_dnn_dk1.py', '--smoke',
       '--dataset', DATASET, '--datasets-dir', DATASETS, '--out-dir', OUT]
print(' '.join(cmd))
subprocess.run(cmd, check=False)

## 4. Measure before committing hours

Time one recalibration on this runtime. Multiply by the number of forecast
days and seeds to get the real cost of the full run before starting it.

The reference figure from a CPU sandbox was **77.6 s** per recalibration.

In [ ]:
import time, pandas as pd
from dnn_dk1 import DNN

hyper_dir = os.path.join(OUT, 'hyperparameters')
data = pd.read_csv(os.path.join(DATASETS, DATASET + '.csv'),
                   index_col=0, parse_dates=True)
data.columns = ['Price'] + [f'Exogenous {i}' for i in range(1, len(data.columns))]

model = DNN(path_hyperparameter_folder=hyper_dir, dataset=DATASET,
            calibration_window=4, seed=1)

day = data.index[-1].normalize()
started = time.time()
_ = model.recalibrate_and_forecast_next_day(data, day)
per_day = time.time() - started

print(f'one recalibration: {per_day:.1f}s')
for days, seeds in ((728, 4), (728, 1), (104, 4)):
    hours = per_day * days * seeds / 3600
    print(f'  {days} days x {seeds} seed(s): {hours:.1f} h')

## 5. The real run

The paper's specification: 1500 hyperparameter evaluations, then 728 test
days recalibrated daily, ensembled over 4 seeds.

**Colab disconnects.** Free and Pro runtimes are reclaimed after idle time
and have a session cap, and a multi-hour run will be interrupted. Writing to
Drive means an interrupted search keeps its trials file — `dnn_dk1.hyperopt`
checkpoints before every evaluation — so re-running with `--skip-hyperopt`
resumes from what survived. The forecast loop has no such checkpointing yet;
run it in chunks with `--begin-test`/`--end-test` until it does.

In [ ]:
cmd = [sys.executable, 'run_dnn_dk1.py',
       '--dataset', DATASET, '--datasets-dir', DATASETS, '--out-dir', OUT,
       '--max-evals', '1500',
       '--seeds', '1,2,3,4']
print(' '.join(cmd))
# subprocess.run(cmd, check=False)   # uncomment when you mean it